In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# --------------------------------------------------
# Paths
# --------------------------------------------------

RAW_FILE = Path(
    "../data/raw/Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv"
)

print("Raw file exists:", RAW_FILE.exists())
print("File:", RAW_FILE)

Raw file exists: True
File: ..\data\raw\Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv


In [2]:
df = pd.read_csv(RAW_FILE)

print("✅ Full dataset loaded")
print("Shape:", df.shape)
print("Memory usage (MB):",
      round(df.memory_usage(deep=True).sum() / 1024**2, 2))

✅ Full dataset loaded
Shape: (1048575, 80)
Memory usage (MB): 753.56


In [3]:
# Standardize labels
df["Label"] = df["Label"].astype(str).str.strip()

# Create binary target
df["Target"] = np.where(
    df["Label"].str.lower().eq("benign"),
    0,
    1
)

print(df[["Label", "Target"]].drop_duplicates())

                     Label  Target
0                   Benign       0
277   DDOS attack-LOIC-UDP       1
3930      DDOS attack-HOIC       1


In [4]:
print("Number of columns:", len(df.columns))
print(df.columns.tolist())

Number of columns: 81
['Dst Port', 'Protocol', 'Timestamp', 'Flow Duration', 'Tot Fwd Pkts', 'Tot Bwd Pkts', 'TotLen Fwd Pkts', 'TotLen Bwd Pkts', 'Fwd Pkt Len Max', 'Fwd Pkt Len Min', 'Fwd Pkt Len Mean', 'Fwd Pkt Len Std', 'Bwd Pkt Len Max', 'Bwd Pkt Len Min', 'Bwd Pkt Len Mean', 'Bwd Pkt Len Std', 'Flow Byts/s', 'Flow Pkts/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Tot', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Tot', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Len', 'Bwd Header Len', 'Fwd Pkts/s', 'Bwd Pkts/s', 'Pkt Len Min', 'Pkt Len Max', 'Pkt Len Mean', 'Pkt Len Std', 'Pkt Len Var', 'FIN Flag Cnt', 'SYN Flag Cnt', 'RST Flag Cnt', 'PSH Flag Cnt', 'ACK Flag Cnt', 'URG Flag Cnt', 'CWE Flag Count', 'ECE Flag Cnt', 'Down/Up Ratio', 'Pkt Size Avg', 'Fwd Seg Size Avg', 'Bwd Seg Size Avg', 'Fwd Byts/b Avg', 'Fwd Pkts/b Avg', 'Fwd 

In [5]:
DROP_COLUMNS = [
    "Timestamp",
    "Label"
]

df_ml = df.drop(columns=DROP_COLUMNS).copy()

print("Original shape:", df.shape)
print("ML dataset shape:", df_ml.shape)

Original shape: (1048575, 81)
ML dataset shape: (1048575, 79)


In [6]:
CONSTANT_FEATURES = [
    "Bwd PSH Flags",
    "Fwd URG Flags",
    "Bwd URG Flags",
    "CWE Flag Count",
    "Fwd Byts/b Avg",
    "Fwd Pkts/b Avg",
    "Fwd Blk Rate Avg",
    "Bwd Byts/b Avg",
    "Bwd Pkts/b Avg",
    "Bwd Blk Rate Avg"
]

df_ml = df_ml.drop(
    columns=CONSTANT_FEATURES,
    errors="ignore"
)

print("Shape after removing constant features:", df_ml.shape)

Shape after removing constant features: (1048575, 69)


In [7]:
numeric_columns = df_ml.select_dtypes(
    include=np.number
).columns

print(
    "Infinite values:",
    np.isinf(
        df_ml[numeric_columns].to_numpy()
    ).sum()
)

print(
    "Missing values:",
    df_ml.isna().sum().sum()
)

Infinite values: 0
Missing values: 0


In [8]:
PROCESSED_FILE = Path(
    "../data/processed/detection_dos_cleaned.csv"
)

df_ml.to_csv(
    PROCESSED_FILE,
    index=False
)

print("✅ Cleaned dataset saved!")
print(PROCESSED_FILE)
print("Final shape:", df_ml.shape)

✅ Cleaned dataset saved!
..\data\processed\detection_dos_cleaned.csv
Final shape: (1048575, 69)
